In [1]:
# ============ Avathon pricing-review: full unattended run ============
import os, subprocess, glob, json, sys
from pathlib import Path

!pip -q uninstall -y torchao

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

DATA = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0].rsplit("/", 1)[0]
OUT, RES = "/kaggle/working/outputs", "/kaggle/working/results"
os.makedirs("/kaggle/working/src", exist_ok=True); os.makedirs(RES, exist_ok=True)
!cp {DATA}/*.py /kaggle/working/src/
os.chdir("/kaggle/working")

# ---- 0. fail fast if the uploaded files are stale ----
tp = Path("src/train.py").read_text(); ep = Path("src/evaluate.py").read_text()
missing = [n for n, ok in [
    ("train.chat_template_ids",    "chat_template_ids"    in tp),
    ("train.detect_target_modules","detect_target_modules" in tp),
    ("evaluate.score_sets",        "score_sets"           in ep)] if not ok]
assert not missing, f"STALE FILES: {missing} -- re-upload the dataset before committing"

# patch in Python (no shell brace escaping)
tp = tp.replace("{lora_cfg_targets := lora.target_modules}", "{lora.target_modules}")
tp = tp.replace('"target_modules": lora.target_modules,',
                '"target_modules": sorted(lora.target_modules),')
Path("src/train.py").write_text(tp)
print("files verified and patched", flush=True)

QWEN  = ["Qwen/Qwen2.5-0.5B-Instruct","Qwen/Qwen2.5-1.5B-Instruct","Qwen/Qwen2.5-3B-Instruct"]
LLAMA = ["meta-llama/Llama-3.2-1B-Instruct","meta-llama/Llama-3.2-3B-Instruct"]

def run(gpu, cmds, log):
    return subprocess.Popen(f"CUDA_VISIBLE_DEVICES={gpu} " + " ; ".join(cmds),
        shell=True, stdout=open(log,"w"), stderr=subprocess.STDOUT)

# ---- 1. baseline losses ----
run(0, [f"python src/baseline_loss.py --model {m} --data-dir {DATA} --results-dir {RES}"
        for m in QWEN + LLAMA], "/kaggle/working/00_baseline.log").wait()
print("baselines done", flush=True)

# ---- 2. fine-tune all five, both GPUs ----
t = lambda m: f"python src/train.py --model {m} --data-dir {DATA} --out-dir {OUT} --results-dir {RES}"
a = run(0, [t(m) for m in QWEN],  "/kaggle/working/gpu0.log")
b = run(1, [t(m) for m in LLAMA], "/kaggle/working/gpu1.log")
a.wait();

files verified and patched
baselines done
